In [3]:
import numpy as np
import torch

# 1. 선형 회귀와 자동 미분 (Linear Regression and Autograd)

이번 챕터에서는 선형 회귀 이론을 이해하고, 파이토치(PyTorch)를 이용하여 선형 회귀 모델을 만들어본다.

## preview
> **데이터에 대한 이해(Data Definition)** : 학습할 데이터에 대해 알아본다.

> **가설(Hypothesis) 수립** : 가설을 수립하는 방법에 대해 알아본다.

> **손실 계산하기(Compute loss)** : 학습 데이터를 이용해 모델을 연속적으로 개선시키는데, 이때 손실(loss)을 이용한다.

> **경사 하강법(Gradient Descent)** : 학습을 위한 핵심 알고리즘인 경사 하강법에 대해 이해한다.

## 데이터에 대한 이해 (Data Definition)

이번 챕터의 예제는 **공부한 시간과 점수의 상관관계**다. 어떤 학생이 1시간 공부했더니 2점, 2시간 공부했더니 4점, 3시간 공부했더니 6점을 맞았다. 그렇다면 내가 4시간을 공부하면 몇 점을 맞을 수 있을까?

이 질문에 답하기 위해 사용하는 데이터를 **훈련 데이터셋(training dataset)**이라 하고, 학습이 끝난 후 모델이 얼마나 잘 동작하는지 판별하는 데이터셋을 **테스트 데이터셋(test dataset)**이라 한다.

모델을 학습시키기 위한 데이터는 파이토치 텐서(torch.tensor)의 형태를 가져야 한다. 그리고 입력과 출력을 각기 다른 텐서에 저장한다. 보편적으로 입력은 x, 출력은 y로 표기한다. 여기서 x_train은 공부한 시간, y_train은 그에 맵핑되는 점수를 의미한다.

In [4]:
x_train = torch.FloatTensor([[1], [2], [3]])
y_train = torch.FloatTensor([[2], [4], [6]])

## 가설 (Hypothesis) 수립

머신 러닝에서 세우는 식을 **가설(Hypothesis)**이라 한다. 선형 회귀란 학습 데이터와 가장 잘 맞는 하나의 직선을 찾는 일이고, 그 직선(가설)은 다음과 같은 형식을 가진다.

$$ H(x) = Wx + b $$

이때 x와 곱해지는 **W를 가중치(Weight)**, **b를 편향(bias)**이라 한다. 직선의 방정식에서 각각 기울기와 y절편에 해당한다.

## 비용 함수 (Cost function)

아래 용어들은 전부 같은 의미로 생각하면 된다.

> 비용 함수(cost function) = 손실 함수(loss function) = 오차 함수(error function) = 목적 함수(objective function)

직선이 데이터를 얼마나 잘 표현하는지는 **오차(error)**로 측정한다. 오차 = 실제값 − 예측값인데, 그냥 더하면 양수와 음수가 상쇄되어 제대로 된 크기를 측정할 수 없다. 그래서 각 오차를 **제곱**한 뒤 더하고, 데이터 개수 n으로 나눠 평균을 구한다. 이를 **평균 제곱 오차(Mean Squared Error, MSE)**라고 한다.

$$ cost(W, b) = \frac{1}{n} \sum_{i=1}^{n} \left[ y^{(i)} - H(x^{(i)}) \right]^2 $$

$cost(W, b)$를 최소가 되게 만드는 W와 b를 구하면, 훈련 데이터를 가장 잘 나타내는 직선을 구할 수 있다.

## 옵티마이저 - 경사 하강법 (Gradient Descent)

비용 함수의 값을 최소로 하는 W와 b를 찾는 데 사용하는 것이 **옵티마이저(Optimizer)** 알고리즘이고, 이 과정을 머신 러닝에서 **학습(training)**이라 부른다. 여기서는 가장 기본적인 **경사 하강법(Gradient Descent)**을 본다. (설명의 편의를 위해 편향 b는 제외하고 $H(x) = Wx$ 로 가정한다.)

cost가 최소가 되는 지점은 접선의 기울기(미분값)가 0이 되는 지점이다. 경사 하강법은 비용 함수를 미분하여 현재 W에서의 접선의 기울기를 구하고, 기울기가 낮은 방향으로 W를 반복해서 변경한다.

$$ W := W - \alpha \frac{\partial}{\partial W} cost(W) $$

- 기울기가 **음수**일 때 → W의 값이 **증가**
- 기울기가 **양수**일 때 → W의 값이 **감소**

두 경우 모두 접선의 기울기가 0인 방향으로 W가 조정된다. 여기서 $\alpha$는 **학습률(learning rate)**로, W를 한 번에 얼마나 크게 변경할지를 결정한다. 학습률이 지나치게 크면 W가 발산하고, 지나치게 작으면 학습 속도가 느려지므로 적당한 값을 찾는 것이 중요하다.

> 가설, 비용 함수, 옵티마이저는 머신 러닝의 포괄적 개념이다. 문제마다 다를 수 있으며, 선형 회귀에 가장 적합한 비용 함수는 평균 제곱 오차, 옵티마이저는 경사 하강법이다.

## 파이토치로 선형 회귀 구현하기

### 기본 셋팅

실습에 필요한 파이토치 도구들을 임포트한다. `torch.manual_seed()`는 코드를 재실행해도 같은 결과가 나오도록 랜덤 시드를 고정한다.

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# 현재 실습하고 있는 파이썬 코드를 재실행해도 다음에도 같은 결과가 나오도록 랜덤 시드(random seed)를 줍니다.
torch.manual_seed(1)

### 변수 선언

훈련 데이터 x_train, y_train을 선언하고 크기(shape)를 확인한다.

In [6]:
x_train = torch.FloatTensor([[1], [2], [3]])
y_train = torch.FloatTensor([[2], [4], [6]])

In [7]:
print(x_train)
print(x_train.shape)

tensor([[1.],
        [2.],
        [3.]])
torch.Size([3, 1])


x_train의 크기가 (3 × 1)임을 알 수 있다.

In [8]:
print(y_train)
print(y_train.shape)

tensor([[2.],
        [4.],
        [6.]])
torch.Size([3, 1])


y_train의 크기도 (3 × 1)이다.

### 가중치와 편향의 초기화

선형 회귀의 목표는 가장 잘 맞는 직선을 정의하는 W와 b를 찾는 것이다. 우선 가중치 W를 0으로 초기화한다. `requires_grad=True`는 이 변수가 학습을 통해 계속 값이 변경되는 변수임을 의미한다.

In [9]:
# 가중치 W를 0으로 초기화하고 학습을 통해 값이 변경되는 변수임을 명시함.
W = torch.zeros(1, requires_grad=True)
# 가중치 W를 출력
print(W)

tensor([0.], requires_grad=True)


마찬가지로 편향 b도 0으로 초기화하고, 학습 대상임을 명시한다.

In [10]:
b = torch.zeros(1, requires_grad=True)
print(b)

tensor([0.], requires_grad=True)


현재 W와 b가 둘 다 0이므로 직선의 방정식은 `y = 0*x + 0`이다. 즉 x에 어떤 값이 들어가도 가설은 0을 예측한다. 아직 적절한 W, b의 값이 아니다.

### 가설 세우기

직선의 방정식 $H(x) = Wx + b$ 에 해당하는 가설을 선언한다.

In [11]:
hypothesis = x_train * W + b
print(hypothesis)

tensor([[0.],
        [0.],
        [0.]], grad_fn=<AddBackward0>)


### 비용 함수 선언하기

앞서 배운 `torch.mean`으로 평균 제곱 오차를 선언한다.

In [12]:
# 앞서 배운 torch.mean으로 평균을 구한다.
cost = torch.mean((hypothesis - y_train) ** 2)
print(cost)

tensor(18.6667, grad_fn=<MeanBackward0>)


### 경사 하강법 구현하기

`SGD`는 경사 하강법의 일종이고, `lr`은 학습률(learning rate)을 의미한다. 학습 대상인 W와 b가 SGD의 입력이 된다.

In [13]:
optimizer = optim.SGD([W, b], lr=0.01)

`optimizer.zero_grad()`로 미분을 통해 얻은 기울기를 0으로 초기화하고, `cost.backward()`로 W, b에 대한 기울기를 계산한 뒤, `optimizer.step()`으로 W와 b를 업데이트한다.

In [14]:
# gradient를 0으로 초기화
optimizer.zero_grad()
# 비용 함수를 미분하여 gradient 계산
cost.backward()
# W와 b를 업데이트
optimizer.step()

### 전체 코드

위 과정을 하나로 합친 전체 코드다.

In [15]:
# 데이터
x_train = torch.FloatTensor([[1], [2], [3]])
y_train = torch.FloatTensor([[2], [4], [6]])
# 모델 초기화
W = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
# optimizer 설정
optimizer = optim.SGD([W, b], lr=0.01)

nb_epochs = 1999 # 원하는만큼 경사 하강법을 반복
for epoch in range(nb_epochs + 1):

    # H(x) 계산
    hypothesis = x_train * W + b

    # cost 계산
    cost = torch.mean((hypothesis - y_train) ** 2)

    # cost로 H(x) 개선
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    # 100번마다 로그 출력
    if epoch % 100 == 0:
        print('Epoch {:4d}/{} W: {:.3f}, b: {:.3f} Cost: {:.6f}'.format(
            epoch, nb_epochs, W.item(), b.item(), cost.item()
        ))

Epoch    0/1999 W: 0.187, b: 0.080 Cost: 18.666666
Epoch  100/1999 W: 1.746, b: 0.578 Cost: 0.048171
Epoch  200/1999 W: 1.800, b: 0.454 Cost: 0.029767
Epoch  300/1999 W: 1.843, b: 0.357 Cost: 0.018394
Epoch  400/1999 W: 1.876, b: 0.281 Cost: 0.011366
Epoch  500/1999 W: 1.903, b: 0.221 Cost: 0.007024
Epoch  600/1999 W: 1.924, b: 0.174 Cost: 0.004340
Epoch  700/1999 W: 1.940, b: 0.136 Cost: 0.002682
Epoch  800/1999 W: 1.953, b: 0.107 Cost: 0.001657
Epoch  900/1999 W: 1.963, b: 0.084 Cost: 0.001024
Epoch 1000/1999 W: 1.971, b: 0.066 Cost: 0.000633
Epoch 1100/1999 W: 1.977, b: 0.052 Cost: 0.000391
Epoch 1200/1999 W: 1.982, b: 0.041 Cost: 0.000242
Epoch 1300/1999 W: 1.986, b: 0.032 Cost: 0.000149
Epoch 1400/1999 W: 1.989, b: 0.025 Cost: 0.000092
Epoch 1500/1999 W: 1.991, b: 0.020 Cost: 0.000057
Epoch 1600/1999 W: 1.993, b: 0.016 Cost: 0.000035
Epoch 1700/1999 W: 1.995, b: 0.012 Cost: 0.000022
Epoch 1800/1999 W: 1.996, b: 0.010 Cost: 0.000013
Epoch 1900/1999 W: 1.997, b: 0.008 Cost: 0.000008

최종 결과를 보면 최적의 W는 2에 가깝고 b는 0에 가깝다. 훈련 데이터가 x_train=[[1],[2],[3]], y_train=[[2],[4],[6]]이므로 실제 정답은 $H(x) = 2x$ (W=2, b=0)인데, 거의 정답을 찾은 셈이다.

## optimizer.zero_grad()가 필요한 이유

파이토치는 미분을 통해 얻은 기울기를 **이전에 계산된 기울기 값에 누적**시키는 특징이 있다. 아래 예제로 확인해본다.

In [16]:
import torch
w = torch.tensor(2.0, requires_grad=True)

nb_epochs = 20
for epoch in range(nb_epochs + 1):
    z = 2*w
    z.backward()
    print('수식을 w로 미분한 값 : {}'.format(w.grad))

수식을 w로 미분한 값 : 2.0
수식을 w로 미분한 값 : 4.0
수식을 w로 미분한 값 : 6.0
수식을 w로 미분한 값 : 8.0
수식을 w로 미분한 값 : 10.0
수식을 w로 미분한 값 : 12.0
수식을 w로 미분한 값 : 14.0
수식을 w로 미분한 값 : 16.0
수식을 w로 미분한 값 : 18.0
수식을 w로 미분한 값 : 20.0
수식을 w로 미분한 값 : 22.0
수식을 w로 미분한 값 : 24.0
수식을 w로 미분한 값 : 26.0
수식을 w로 미분한 값 : 28.0
수식을 w로 미분한 값 : 30.0
수식을 w로 미분한 값 : 32.0
수식을 w로 미분한 값 : 34.0
수식을 w로 미분한 값 : 36.0
수식을 w로 미분한 값 : 38.0
수식을 w로 미분한 값 : 40.0
수식을 w로 미분한 값 : 42.0


미분값인 2가 계속 누적되는 것을 볼 수 있다. 그렇기 때문에 `optimizer.zero_grad()`로 미분값을 매번 0으로 초기화해줘야 한다.

## torch.manual_seed()를 하는 이유

`torch.manual_seed()`는 난수 발생 순서와 값을 동일하게 보장한다. 따라서 다른 컴퓨터에서 실행해도 같은 결과를 얻을 수 있다. 랜덤 시드가 3일 때 두 번 난수를 발생시켜보고, 다른 시드를 쓴 뒤 다시 3으로 돌아오면 같은 값이 나오는지 확인한다.

In [17]:
import torch
torch.manual_seed(3)
print('랜덤 시드가 3일 때')
for i in range(1,3):
    print(torch.rand(1))

랜덤 시드가 3일 때
tensor([0.0043])
tensor([0.1056])


In [18]:
torch.manual_seed(5)
print('랜덤 시드가 5일 때')
for i in range(1,3):
    print(torch.rand(1))

랜덤 시드가 5일 때
tensor([0.8303])
tensor([0.1261])


In [19]:
torch.manual_seed(3)
print('랜덤 시드가 다시 3일 때')
for i in range(1,3):
    print(torch.rand(1))

랜덤 시드가 다시 3일 때
tensor([0.0043])
tensor([0.1056])


시드를 다시 3으로 돌리니 처음과 동일한 난수가 나온다. 난수 발생 순서가 초기화되기 때문이다.

## 자동 미분 (Autograd) 실습하기

텐서에는 `requires_grad` 속성이 있다. 이를 True로 설정하면 자동 미분 기능이 적용되어, 연산 시 계산 그래프가 생성되고 `backward()`를 호출하면 그래프로부터 자동으로 미분이 계산된다. 임의로 $2w^2 + 5$ 라는 식을 세우고 w에 대해 미분해본다.

우선 값이 2인 스칼라 텐서 w를 선언하고 `requires_grad=True`로 설정한다.

In [20]:
import torch
w = torch.tensor(2.0, requires_grad=True)

이제 수식을 정의한다.

In [21]:
y = w**2
z = 2*y + 5

`.backward()`를 호출하면 z의 w에 대한 기울기를 계산한다.

In [22]:
z.backward()

`w.grad`를 출력하면 w로 미분한 값이 저장되어 있다. ($\frac{dz}{dw} = 4w = 8$)

In [23]:
print('수식을 w로 미분한 값 : {}'.format(w.grad))

수식을 w로 미분한 값 : 8.0


# 2. 다중 선형 회귀 (Multivariable Linear Regression)

앞서 배운 x가 1개인 선형 회귀를 **단순 선형 회귀**라 한다. 이번에는 다수의 x로부터 y를 예측하는 **다중 선형 회귀(Multivariable Linear Regression)**를 이해한다.

## 데이터에 대한 이해 (Data Definition)

3개의 퀴즈 점수(x1, x2, x3)로부터 최종 점수(y)를 예측하는 모델이다. 단순 선형 회귀와 달리 독립 변수 x가 3개다.

| Quiz 1 (x1) | Quiz 2 (x2) | Quiz 3 (x3) | Final (y) |
|---|---|---|---|
| 73 | 80 | 75 | 152 |
| 93 | 88 | 93 | 185 |
| 89 | 91 | 80 | 180 |
| 96 | 98 | 100 | 196 |
| 73 | 66 | 70 | 142 |

독립 변수가 3개이므로 수식은 다음과 같다.

$$ H(x) = w_1 x_1 + w_2 x_2 + w_3 x_3 + b $$

## 파이토치로 구현하기

우선 필요한 도구들을 임포트하고 랜덤 시드를 고정한다.

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(1)

이번에는 x가 3개이므로 x_train도 3개 선언한다.

In [25]:
# 훈련 데이터
x1_train = torch.FloatTensor([[73], [93], [89], [96], [73]])
x2_train = torch.FloatTensor([[80], [88], [91], [98], [66]])
x3_train = torch.FloatTensor([[75], [93], [90], [100], [70]])
y_train = torch.FloatTensor([[152], [185], [180], [196], [142]])

가중치 w도 3개 선언해주어야 한다. 가설을 선언하는 부분에서도 x_train의 개수만큼 w와 곱해주도록 작성한다.

In [26]:
# 가중치 w와 편향 b 초기화
w1 = torch.zeros(1, requires_grad=True)
w2 = torch.zeros(1, requires_grad=True)
w3 = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# optimizer 설정
optimizer = optim.SGD([w1, w2, w3, b], lr=1e-5)

nb_epochs = 1000
for epoch in range(nb_epochs + 1):

    # H(x) 계산
    hypothesis = x1_train * w1 + x2_train * w2 + x3_train * w3 + b

    # cost 계산
    cost = torch.mean((hypothesis - y_train) ** 2)

    # cost로 H(x) 개선
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    # 100번마다 로그 출력
    if epoch % 100 == 0:
        print('Epoch {:4d}/{} w1: {:.3f} w2: {:.3f} w3: {:.3f} b: {:.3f} Cost: {:.6f}'.format(
            epoch, nb_epochs, w1.item(), w2.item(), w3.item(), b.item(), cost.item()
        ))

Epoch    0/1000 w1: 0.294 w2: 0.294 w3: 0.297 b: 0.003 Cost: 29661.800781
Epoch  100/1000 w1: 0.674 w2: 0.661 w3: 0.676 b: 0.008 Cost: 1.563628
Epoch  200/1000 w1: 0.679 w2: 0.655 w3: 0.677 b: 0.008 Cost: 1.497595
Epoch  300/1000 w1: 0.684 w2: 0.649 w3: 0.677 b: 0.008 Cost: 1.435044
Epoch  400/1000 w1: 0.689 w2: 0.643 w3: 0.678 b: 0.008 Cost: 1.375726
Epoch  500/1000 w1: 0.694 w2: 0.638 w3: 0.678 b: 0.009 Cost: 1.319507
Epoch  600/1000 w1: 0.699 w2: 0.633 w3: 0.679 b: 0.009 Cost: 1.266222
Epoch  700/1000 w1: 0.704 w2: 0.627 w3: 0.679 b: 0.009 Cost: 1.215703
Epoch  800/1000 w1: 0.709 w2: 0.622 w3: 0.679 b: 0.009 Cost: 1.167810
Epoch  900/1000 w1: 0.713 w2: 0.617 w3: 0.680 b: 0.009 Cost: 1.122429
Epoch 1000/1000 w1: 0.718 w2: 0.613 w3: 0.680 b: 0.009 Cost: 1.079390


## 벡터와 행렬 연산으로 바꾸기

위 방식은 비효율적이다. 만약 x의 개수가 1,000개라면 x_train과 w를 각각 1,000개씩, 총 2,000개의 변수를 선언해야 하고 가설의 곱셈 항도 1,000개를 작성해야 한다. 이를 해결하기 위해 **행렬 곱셈 연산(벡터의 내적)**을 사용한다.

> 행렬의 곱셈 과정에서 이루어지는 벡터 연산을 **벡터의 내적(Dot Product)**이라고 한다. 예) `1×7 + 2×9 + 3×11 = 58`

### 벡터 연산으로 이해하기

$H(X) = w_1 x_1 + w_2 x_2 + w_3 x_3$ 는 두 벡터의 내적으로 표현할 수 있다. 두 벡터를 각각 X와 W로 표현하면 가설은 다음과 같다.

$$ H(X) = XW $$

x의 개수가 3개였음에도 이제는 X와 W라는 두 개의 변수로 표현된다.

### 행렬 연산으로 이해하기

전체 훈련 데이터의 개수를 셀 수 있는 1개의 단위를 **샘플(sample)**이라 하고(현재 5개), 각 샘플에서 y를 결정하는 각각의 독립 변수 x를 **특성(feature)**이라 한다(현재 3개). 독립 변수 x들을 (샘플 수 × 특성 수) = (5 × 3) 크기의 행렬 X로 표현하고 가중치 벡터 W(3 × 1)를 곱하면 $H(X) = XW$ 가 된다. 여기에 편향 벡터 B를 더하면 전체 가설이 된다.

$$ H(X) = XW + B $$

이처럼 벡터와 행렬 연산은 식을 간단하게 해줄 뿐 아니라, 다수 샘플의 병렬 연산이므로 속도의 이점도 가진다.

## 행렬 연산을 고려하여 파이토치로 구현하기

이번에는 훈련 데이터를 행렬로 선언한다. x_train 하나에 모든 샘플을 전부 선언했다. 즉 (5 × 3) 행렬 X를 선언한 것이다.

In [27]:
x_train = torch.FloatTensor([[73, 80, 75],
                             [93, 88, 93],
                             [89, 91, 80],
                             [96, 98, 100],
                             [73, 66, 70]])
y_train = torch.FloatTensor([[152], [185], [180], [196], [142]])

In [28]:
print(x_train.shape)
print(y_train.shape)

torch.Size([5, 3])
torch.Size([5, 1])


각각 (5 × 3) 행렬과 (5 × 1) 행렬(벡터)이다. 이제 가중치 W와 편향 b를 선언한다. 주목할 점은 W의 크기가 (3 × 1)이라는 점이다. 행렬곱이 성립하려면 좌측 행렬의 열 크기와 우측 행렬의 행 크기가 일치해야 하는데, X가 (5 × 3), W가 (3 × 1)이므로 행렬곱이 가능하다.

In [29]:
# 가중치와 편향 선언
W = torch.zeros((3, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

가설을 행렬곱(`matmul`)으로 간단히 정의한다. 독립 변수의 수를 나중에 늘리거나 줄여도 이 코드는 수정할 필요가 없다.

In [30]:
hypothesis = x_train.matmul(W) + b

이를 반영한 전체 코드는 다음과 같다.

In [31]:
x_train = torch.FloatTensor([[73, 80, 75],
                             [93, 88, 93],
                             [89, 91, 80],
                             [96, 98, 100],
                             [73, 66, 70]])
y_train = torch.FloatTensor([[152], [185], [180], [196], [142]])
# 모델 초기화
W = torch.zeros((3, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)
# optimizer 설정
optimizer = optim.SGD([W, b], lr=1e-5)

nb_epochs = 20
for epoch in range(nb_epochs + 1):

    # H(x) 계산
    # 편향 b는 브로드 캐스팅되어 각 샘플에 더해집니다.
    hypothesis = x_train.matmul(W) + b

    # cost 계산
    cost = torch.mean((hypothesis - y_train) ** 2)

    # cost로 H(x) 개선
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    print('Epoch {:4d}/{} hypothesis: {} Cost: {:.6f}'.format(
        epoch, nb_epochs, hypothesis.squeeze().detach(), cost.item()
    ))

Epoch    0/20 hypothesis: tensor([0., 0., 0., 0., 0.]) Cost: 29661.800781
Epoch    1/20 hypothesis: tensor([66.7178, 80.1701, 76.1025, 86.0194, 61.1565]) Cost: 9537.694336
Epoch    2/20 hypothesis: tensor([104.5421, 125.6208, 119.2478, 134.7862,  95.8280]) Cost: 3069.590088
Epoch    3/20 hypothesis: tensor([125.9858, 151.3882, 143.7087, 162.4333, 115.4844]) Cost: 990.670288
Epoch    4/20 hypothesis: tensor([138.1429, 165.9963, 157.5768, 178.1071, 126.6283]) Cost: 322.481873
Epoch    5/20 hypothesis: tensor([145.0350, 174.2780, 165.4395, 186.9928, 132.9461]) Cost: 107.717064
Epoch    6/20 hypothesis: tensor([148.9423, 178.9730, 169.8976, 192.0301, 136.5279]) Cost: 38.687496
Epoch    7/20 hypothesis: tensor([151.1574, 181.6346, 172.4254, 194.8856, 138.5585]) Cost: 16.499043
Epoch    8/20 hypothesis: tensor([152.4131, 183.1435, 173.8590, 196.5043, 139.7097]) Cost: 9.365656
Epoch    9/20 hypothesis: tensor([153.1250, 183.9988, 174.6723, 197.4217, 140.3625]) Cost: 7.071114
Epoch   10/20 hyp

학습이 끝난 모델에 임의의 입력 값을 넣어 예측해본다.

In [32]:
# 임의의 입력 값에 대한 예측
with torch.no_grad():
    new_input = torch.FloatTensor([[75, 85, 72]]) # 예측하고 싶은 임의의 입력
    prediction = new_input.matmul(W) + b
    print('Predicted value for input {}: {}'.format(new_input.squeeze().tolist(), prediction.item()))

Predicted value for input [75.0, 85.0, 72.0]: 156.8051300048828


`with torch.no_grad():` 블록 안에서 수행되는 모든 연산은 역전파(기울기 계산)를 비활성화한다. 예측할 때는 가중치를 업데이트할 필요가 없으므로, 메모리와 계산 자원을 절약하기 위해 사용한다.

# 3. nn.Module과 클래스로 구현하기

이전까지는 가설과 비용 함수를 직접 정의해 선형 회귀를 구현했다. 이번에는 파이토치에서 이미 구현되어 제공되는 함수들을 불러와 더 쉽게 구현한다. 예를 들어 선형 회귀 모델은 `nn.Linear()`, 평균 제곱 오차는 `nn.functional.mse_loss()`로 구현되어 있다.

```python
import torch.nn as nn
model = nn.Linear(input_dim, output_dim)
```

```python
import torch.nn.functional as F
cost = F.mse_loss(prediction, y_train)
```

## 단순 선형 회귀 구현하기

필요한 도구들을 임포트하고, $y = 2x$ 를 가정한 데이터를 선언한다. 정답이 W=2, b=0임을 이미 알고 있는 상태이고, 모델이 이 값을 제대로 찾아내도록 하는 것이 목표다.

In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1)

In [34]:
# 데이터
x_train = torch.FloatTensor([[1], [2], [3]])
y_train = torch.FloatTensor([[2], [4], [6]])

`nn.Linear()`는 입력의 차원과 출력의 차원을 인수로 받는다. 단순 선형 회귀이므로 입력/출력 차원 모두 1이다.

In [35]:
# 모델을 선언 및 초기화. 단순 선형 회귀이므로 input_dim=1, output_dim=1.
model = nn.Linear(1,1)

model에는 가중치 W와 편향 b가 저장되어 있다. `model.parameters()`로 불러올 수 있다.

In [36]:
print(list(model.parameters()))

[Parameter containing:
tensor([[0.5153]], requires_grad=True), Parameter containing:
tensor([-0.4414], requires_grad=True)]


2개의 값이 출력되는데 첫번째가 W, 두번째가 b다. 둘 다 현재는 랜덤 초기화되어 있고, 학습 대상이므로 `requires_grad=True`로 되어 있다. 이제 옵티마이저를 정의한다.

In [37]:
# optimizer 설정. 경사 하강법 SGD를 사용하고 learning rate를 의미하는 lr은 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [38]:
# 전체 훈련 데이터에 대해 경사 하강법을 2,000회 반복
nb_epochs = 2000
for epoch in range(nb_epochs+1):

    # H(x) 계산
    prediction = model(x_train)

    # cost 계산
    cost = F.mse_loss(prediction, y_train) # <== 파이토치에서 제공하는 평균 제곱 오차 함수

    # cost로 H(x) 개선하는 부분
    # gradient를 0으로 초기화
    optimizer.zero_grad()
    # 비용 함수를 미분하여 gradient 계산
    cost.backward() # backward 연산
    # W와 b를 업데이트
    optimizer.step()

    if epoch % 100 == 0:
        # 100번마다 로그 출력
        print('Epoch {:4d}/{} Cost: {:.6f}'.format(
            epoch, nb_epochs, cost.item()
        ))

Epoch    0/2000 Cost: 13.103541
Epoch  100/2000 Cost: 0.002791
Epoch  200/2000 Cost: 0.001724
Epoch  300/2000 Cost: 0.001066
Epoch  400/2000 Cost: 0.000658
Epoch  500/2000 Cost: 0.000407
Epoch  600/2000 Cost: 0.000251
Epoch  700/2000 Cost: 0.000155
Epoch  800/2000 Cost: 0.000096
Epoch  900/2000 Cost: 0.000059
Epoch 1000/2000 Cost: 0.000037
Epoch 1100/2000 Cost: 0.000023
Epoch 1200/2000 Cost: 0.000014
Epoch 1300/2000 Cost: 0.000009
Epoch 1400/2000 Cost: 0.000005
Epoch 1500/2000 Cost: 0.000003
Epoch 1600/2000 Cost: 0.000002
Epoch 1700/2000 Cost: 0.000001
Epoch 1800/2000 Cost: 0.000001
Epoch 1900/2000 Cost: 0.000000
Epoch 2000/2000 Cost: 0.000000


학습이 완료되었다. x에 임의의 값 4를 넣어 예측값을 확인한다. $y = 2x$ 이므로 8에 가까운 값이 나와야 한다.

In [39]:
# 임의의 입력 4를 선언
new_var = torch.FloatTensor([[4.0]])
# 입력한 값 4에 대해서 예측값 y를 리턴받아서 pred_y에 저장
pred_y = model(new_var) # forward 연산
# y = 2x 이므로 입력이 4라면 y가 8에 가까운 값이 나와야 제대로 학습이 된 것
print("훈련 후 입력이 4일 때의 예측값 :", pred_y)

훈련 후 입력이 4일 때의 예측값 : tensor([[7.9989]], grad_fn=<AddmmBackward0>)


학습 후의 W와 b의 값을 출력해본다. W가 2에, b가 0에 가까운 것을 볼 수 있다.

In [40]:
print(list(model.parameters()))

[Parameter containing:
tensor([[1.9994]], requires_grad=True), Parameter containing:
tensor([0.0014], requires_grad=True)]


> 식에 입력 x로부터 예측된 y를 얻는 것을 **forward 연산**이라 한다. (`prediction = model(x_train)`, `pred_y = model(new_var)`)

> 학습 과정에서 비용 함수를 미분하여 기울기를 구하는 것을 **backward 연산**이라 한다. (`cost.backward()`)

## 다중 선형 회귀 구현하기

이제 `nn.Linear()`와 `nn.functional.mse_loss()`로 다중 선형 회귀를 구현한다. 코드 자체는 거의 달라지지 않고, `nn.Linear()`의 인자값과 학습률만 조절한다.

In [41]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1)

In [42]:
# 데이터
x_train = torch.FloatTensor([[73, 80, 75],
                             [93, 88, 93],
                             [89, 91, 90],
                             [96, 98, 100],
                             [73, 66, 70]])
y_train = torch.FloatTensor([[152], [185], [180], [196], [142]])

3개의 입력 x에 대해 하나의 출력 y를 가지므로 입력 차원은 3, 출력 차원은 1이다.

In [43]:
# 모델을 선언 및 초기화. 다중 선형 회귀이므로 input_dim=3, output_dim=1.
model = nn.Linear(3,1)

In [44]:
print(list(model.parameters()))

[Parameter containing:
tensor([[ 0.2975, -0.2548, -0.1119]], requires_grad=True), Parameter containing:
tensor([0.2710], requires_grad=True)]


첫번째 출력이 3개의 w, 두번째가 b다. 옵티마이저를 정의한다. 학습률은 1e-5(=0.00001)로 정하는데, 0.01로 하지 않는 이유는 기울기가 발산하기 때문이다.

In [45]:
optimizer = torch.optim.SGD(model.parameters(), lr=1e-5)

이하 코드는 단순 선형 회귀를 구현했을 때와 동일하다.

In [46]:
nb_epochs = 2000
for epoch in range(nb_epochs+1):

    # H(x) 계산
    prediction = model(x_train)
    # model(x_train)은 model.forward(x_train)와 동일함.

    # cost 계산
    cost = F.mse_loss(prediction, y_train) # <== 파이토치에서 제공하는 평균 제곱 오차 함수

    # cost로 H(x) 개선하는 부분
    # gradient를 0으로 초기화
    optimizer.zero_grad()
    # 비용 함수를 미분하여 gradient 계산
    cost.backward()
    # W와 b를 업데이트
    optimizer.step()

    if epoch % 100 == 0:
        # 100번마다 로그 출력
        print('Epoch {:4d}/{} Cost: {:.6f}'.format(
            epoch, nb_epochs, cost.item()
        ))

Epoch    0/2000 Cost: 31667.597656
Epoch  100/2000 Cost: 0.225993
Epoch  200/2000 Cost: 0.223911
Epoch  300/2000 Cost: 0.221941
Epoch  400/2000 Cost: 0.220059
Epoch  500/2000 Cost: 0.218271
Epoch  600/2000 Cost: 0.216575
Epoch  700/2000 Cost: 0.214950
Epoch  800/2000 Cost: 0.213413
Epoch  900/2000 Cost: 0.211952
Epoch 1000/2000 Cost: 0.210560
Epoch 1100/2000 Cost: 0.209232
Epoch 1200/2000 Cost: 0.207967
Epoch 1300/2000 Cost: 0.206761
Epoch 1400/2000 Cost: 0.205619
Epoch 1500/2000 Cost: 0.204522
Epoch 1600/2000 Cost: 0.203484
Epoch 1700/2000 Cost: 0.202485
Epoch 1800/2000 Cost: 0.201542
Epoch 1900/2000 Cost: 0.200635
Epoch 2000/2000 Cost: 0.199769


In [47]:
# 임의의 입력 [73, 80, 75]를 선언
new_var = torch.FloatTensor([[73, 80, 75]])
# 입력한 값 [73, 80, 75]에 대해서 예측값 y를 리턴받아서 pred_y에 저장
pred_y = model(new_var)
print("훈련 후 입력이 73, 80, 75일 때의 예측값 :", pred_y)

훈련 후 입력이 73, 80, 75일 때의 예측값 : tensor([[151.2305]], grad_fn=<AddmmBackward0>)


73, 80, 75는 훈련 데이터로 쓰였던 값이고 당시 y는 152였는데, 예측값이 151 부근으로 나오는 것으로 보아 어느 정도 최적화된 것이다.

In [48]:
print(list(model.parameters()))

[Parameter containing:
tensor([[0.9778, 0.4539, 0.5768]], requires_grad=True), Parameter containing:
tensor([0.2802], requires_grad=True)]


## 모델을 클래스로 구현하기

파이토치의 대부분의 구현체는 모델을 생성할 때 **클래스(Class)**를 사용한다. 앞서 구현한 코드와 다른 점은 오직 클래스로 모델을 구현했다는 점뿐이다. 먼저 단순 선형 회귀 `model = nn.Linear(1,1)` 을 클래스로 구현하면 다음과 같다.

In [49]:
class LinearRegressionModel(nn.Module): # torch.nn.Module을 상속받는 파이썬 클래스
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1) # 단순 선형 회귀이므로 input_dim=1, output_dim=1.

    def forward(self, x):
        return self.linear(x)

model = LinearRegressionModel()

> 클래스 형태의 모델은 `nn.Module`을 상속받는다.
> - `__init__()` : 모델의 구조와 동작을 정의하는 생성자. 객체가 생성될 때 자동 호출된다. `super()`를 부르면 이 클래스는 `nn.Module`의 속성들을 가지고 초기화된다.
> - `forward()` : 모델이 학습 데이터를 입력받아 forward 연산을 진행하는 함수. `model(입력 데이터)` 형식으로 객체를 호출하면 자동으로 실행된다.

앞서 다중 선형 회귀 `model = nn.Linear(3,1)` 을 클래스로 구현하면 다음과 같다.

In [50]:
class MultivariateLinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(3, 1) # 다중 선형 회귀이므로 input_dim=3, output_dim=1.

    def forward(self, x):
        return self.linear(x)

model = MultivariateLinearRegressionModel()

## 단순 선형 회귀 클래스로 구현하기

이제 모델을 클래스로 구현한 전체 코드를 본다. 달라진 점은 모델을 클래스로 구현했다는 점뿐이고, 다른 코드는 전부 동일하다.

In [51]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1)

In [52]:
# 데이터
x_train = torch.FloatTensor([[1], [2], [3]])
y_train = torch.FloatTensor([[2], [4], [6]])

In [53]:
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)

model = LinearRegressionModel()

In [54]:
# optimizer 설정. 경사 하강법 SGD를 사용하고 learning rate를 의미하는 lr은 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [55]:
# 전체 훈련 데이터에 대해 경사 하강법을 2,000회 반복
nb_epochs = 2000
for epoch in range(nb_epochs+1):

    # H(x) 계산
    prediction = model(x_train)

    # cost 계산
    cost = F.mse_loss(prediction, y_train) # <== 파이토치에서 제공하는 평균 제곱 오차 함수

    # cost로 H(x) 개선하는 부분
    # gradient를 0으로 초기화
    optimizer.zero_grad()
    # 비용 함수를 미분하여 gradient 계산
    cost.backward() # backward 연산
    # W와 b를 업데이트
    optimizer.step()

    if epoch % 100 == 0:
        # 100번마다 로그 출력
        print('Epoch {:4d}/{} Cost: {:.6f}'.format(
            epoch, nb_epochs, cost.item()
        ))

Epoch    0/2000 Cost: 13.103541
Epoch  100/2000 Cost: 0.002791
Epoch  200/2000 Cost: 0.001724
Epoch  300/2000 Cost: 0.001066
Epoch  400/2000 Cost: 0.000658
Epoch  500/2000 Cost: 0.000407
Epoch  600/2000 Cost: 0.000251
Epoch  700/2000 Cost: 0.000155
Epoch  800/2000 Cost: 0.000096
Epoch  900/2000 Cost: 0.000059
Epoch 1000/2000 Cost: 0.000037
Epoch 1100/2000 Cost: 0.000023
Epoch 1200/2000 Cost: 0.000014
Epoch 1300/2000 Cost: 0.000009
Epoch 1400/2000 Cost: 0.000005
Epoch 1500/2000 Cost: 0.000003
Epoch 1600/2000 Cost: 0.000002
Epoch 1700/2000 Cost: 0.000001
Epoch 1800/2000 Cost: 0.000001
Epoch 1900/2000 Cost: 0.000000
Epoch 2000/2000 Cost: 0.000000


## 다중 선형 회귀 클래스로 구현하기

마찬가지로 다중 선형 회귀를 클래스로 구현한 전체 코드다.

In [56]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1)

In [57]:
# 데이터
x_train = torch.FloatTensor([[73, 80, 75],
                             [93, 88, 93],
                             [89, 91, 90],
                             [96, 98, 100],
                             [73, 66, 70]])
y_train = torch.FloatTensor([[152], [185], [180], [196], [142]])

In [58]:
class MultivariateLinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(3, 1) # 다중 선형 회귀이므로 input_dim=3, output_dim=1.

    def forward(self, x):
        return self.linear(x)

model = MultivariateLinearRegressionModel()

In [59]:
optimizer = torch.optim.SGD(model.parameters(), lr=1e-5)

In [60]:
nb_epochs = 2000
for epoch in range(nb_epochs+1):

    # H(x) 계산
    prediction = model(x_train)
    # model(x_train)은 model.forward(x_train)와 동일함.

    # cost 계산
    cost = F.mse_loss(prediction, y_train) # <== 파이토치에서 제공하는 평균 제곱 오차 함수

    # cost로 H(x) 개선하는 부분
    # gradient를 0으로 초기화
    optimizer.zero_grad()
    # 비용 함수를 미분하여 gradient 계산
    cost.backward()
    # W와 b를 업데이트
    optimizer.step()

    if epoch % 100 == 0:
        # 100번마다 로그 출력
        print('Epoch {:4d}/{} Cost: {:.6f}'.format(
            epoch, nb_epochs, cost.item()
        ))

Epoch    0/2000 Cost: 31667.597656
Epoch  100/2000 Cost: 0.225993
Epoch  200/2000 Cost: 0.223911
Epoch  300/2000 Cost: 0.221941
Epoch  400/2000 Cost: 0.220059
Epoch  500/2000 Cost: 0.218271
Epoch  600/2000 Cost: 0.216575
Epoch  700/2000 Cost: 0.214950
Epoch  800/2000 Cost: 0.213413
Epoch  900/2000 Cost: 0.211952
Epoch 1000/2000 Cost: 0.210560
Epoch 1100/2000 Cost: 0.209232
Epoch 1200/2000 Cost: 0.207967
Epoch 1300/2000 Cost: 0.206761
Epoch 1400/2000 Cost: 0.205619
Epoch 1500/2000 Cost: 0.204522
Epoch 1600/2000 Cost: 0.203484
Epoch 1700/2000 Cost: 0.202485
Epoch 1800/2000 Cost: 0.201542
Epoch 1900/2000 Cost: 0.200635
Epoch 2000/2000 Cost: 0.199769


> 위와 같은 클래스를 사용한 모델 구현 형식은 대부분의 파이토치 구현체에서 사용하는 방식이므로 반드시 숙지할 필요가 있다.

# 4. 미니 배치와 데이터 로더 (Mini Batch and DataLoader)

이번 챕터는 선형 회귀에 한정되는 내용은 아니다. 데이터를 로드하는 방법과 **미니 배치 경사 하강법(Minibatch Gradient Descent)**에 대해 학습한다.

## 미니 배치와 배치 크기 (Mini Batch and Batch Size)

앞서 다중 선형 회귀에서 사용한 데이터를 상기해본다.

In [61]:
x_train = torch.FloatTensor([[73, 80, 75],
                             [93, 88, 93],
                             [89, 91, 90],
                             [96, 98, 100],
                             [73, 66, 70]])
y_train = torch.FloatTensor([[152], [185], [180], [196], [142]])

위 데이터의 샘플은 5개로 매우 적다. 만약 데이터가 수십만 개라면 전체 데이터에 대해 경사 하강법을 수행하는 것은 매우 느리고 계산량도 많으며, 메모리 한계로 불가능할 수도 있다. 그래서 전체 데이터를 더 작은 단위로 나누어 학습하는데, 이 단위를 **미니 배치(Mini Batch)**라고 한다.

미니 배치 학습은 미니 배치만큼만 가져가 비용(cost)을 계산하고 경사 하강법을 수행하며, 마지막 미니 배치까지 반복한다. 전체 데이터에 대한 학습이 1회 끝나면 1 **에포크(Epoch)**가 끝난다. 미니 배치의 크기를 **배치 크기(batch size)**라고 한다.

> - 전체 데이터에 대해 한 번에 경사 하강법을 수행하는 방법을 **배치 경사 하강법**이라 한다. 가중치가 최적값에 수렴하는 과정이 매우 안정적이지만 계산량이 많다.
> - 미니 배치 단위로 경사 하강법을 수행하는 방법을 **미니 배치 경사 하강법**이라 한다. 전체 데이터의 일부만 보고 수행하므로 수렴 과정에서 값이 조금 헤매기도 하지만 훈련 속도가 빠르다.
> - 배치 크기는 보통 **2의 제곱수**(2, 4, 8, 16, 32, 64...)를 사용한다. CPU와 GPU의 메모리가 2의 배수이므로 데이터 송수신 효율을 높일 수 있기 때문이다.

## 이터레이션 (Iteration)

**이터레이션(iteration)**은 한 번의 에포크 내에서 이루어지는 매개변수(W, b) 업데이트 횟수다. 예를 들어 전체 데이터가 2,000이고 배치 크기가 200이라면 이터레이션의 수는 10이다. 즉 한 번의 에포크당 매개변수 업데이트가 10번 이루어진다.

## 데이터 로드하기 (Data Load)

파이토치는 데이터를 쉽게 다루도록 **데이터셋(Dataset)**과 **데이터로더(DataLoader)**를 제공한다. 이를 사용하면 미니 배치 학습, 데이터 셔플(shuffle), 병렬 처리까지 간단히 수행할 수 있다. 기본 사용법은 Dataset을 정의하고 이를 DataLoader에 전달하는 것이다. 여기서는 텐서를 입력받아 Dataset 형태로 변환해주는 `TensorDataset`을 사용한다.

실습에 필요한 도구들을 임포트한다.

In [62]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import TensorDataset # 텐서데이터셋
from torch.utils.data import DataLoader # 데이터로더

TensorDataset은 기본적으로 텐서를 입력으로 받는다. 텐서 형태로 데이터를 정의한다.

In [63]:
x_train = torch.FloatTensor([[73, 80, 75],
                             [93, 88, 93],
                             [89, 91, 90],
                             [96, 98, 100],
                             [73, 66, 70]])
y_train = torch.FloatTensor([[152], [185], [180], [196], [142]])

이를 TensorDataset의 입력으로 사용하여 dataset으로 저장한다.

In [64]:
dataset = TensorDataset(x_train, y_train)

데이터셋을 만들었으면 데이터로더를 사용할 수 있다. 데이터로더는 기본적으로 2개의 인자(데이터셋, 미니 배치 크기)를 받는다. 미니 배치 크기는 통상 2의 배수를 쓴다. 추가로 많이 쓰는 인자로 `shuffle`이 있는데, `shuffle=True`로 하면 에포크마다 데이터셋을 섞어 학습 순서를 바꾼다. 모델이 데이터셋의 순서에 익숙해지는 것을 방지하기 위해 권장한다.

In [65]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

이제 모델과 옵티마이저를 설계한다.

In [66]:
model = nn.Linear(3,1)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-5)

이제 훈련을 진행한다. 외부 반복문은 에포크를, 내부 반복문은 데이터로더에서 미니 배치 단위로 데이터를 가져와 학습시킨다. `batch_idx`와 `samples`가 어떻게 동작하는지 궁금하면 주석을 해제하고 실행해본다.

In [67]:
nb_epochs = 20
for epoch in range(nb_epochs + 1):
    for batch_idx, samples in enumerate(dataloader):
        # print(batch_idx)
        # print(samples)
        x_train, y_train = samples
        # H(x) 계산
        prediction = model(x_train)

        # cost 계산
        cost = F.mse_loss(prediction, y_train)

        # cost로 H(x) 계산
        optimizer.zero_grad()
        cost.backward()
        optimizer.step()

        print('Epoch {:4d}/{} Batch {}/{} Cost: {:.6f}'.format(
            epoch, nb_epochs, batch_idx+1, len(dataloader),
            cost.item()
        ))

Epoch    0/20 Batch 1/3 Cost: 40394.078125
Epoch    0/20 Batch 2/3 Cost: 10625.847656
Epoch    0/20 Batch 3/3 Cost: 5451.569824
Epoch    1/20 Batch 1/3 Cost: 937.502686
Epoch    1/20 Batch 2/3 Cost: 265.944641
Epoch    1/20 Batch 3/3 Cost: 244.422516
Epoch    2/20 Batch 1/3 Cost: 3.523124
Epoch    2/20 Batch 2/3 Cost: 55.838848
Epoch    2/20 Batch 3/3 Cost: 2.371109
Epoch    3/20 Batch 1/3 Cost: 21.615038
Epoch    3/20 Batch 2/3 Cost: 1.536160
Epoch    3/20 Batch 3/3 Cost: 23.620941
Epoch    4/20 Batch 1/3 Cost: 12.848326
Epoch    4/20 Batch 2/3 Cost: 17.578579
Epoch    4/20 Batch 3/3 Cost: 18.382185
Epoch    5/20 Batch 1/3 Cost: 11.401794
Epoch    5/20 Batch 2/3 Cost: 20.961008
Epoch    5/20 Batch 3/3 Cost: 28.939528
Epoch    6/20 Batch 1/3 Cost: 14.807928
Epoch    6/20 Batch 2/3 Cost: 2.599214
Epoch    6/20 Batch 3/3 Cost: 33.501408
Epoch    7/20 Batch 1/3 Cost: 16.869902
Epoch    7/20 Batch 2/3 Cost: 24.529007
Epoch    7/20 Batch 3/3 Cost: 10.461083
Epoch    8/20 Batch 1/3 Cost: 8.6

Cost 값이 점차 작아진다. 모델에 임의의 값을 넣어 예측값을 확인한다.

In [68]:
# 임의의 입력 [73, 80, 75]를 선언
new_var = torch.FloatTensor([[73, 80, 75]])
# 입력한 값 [73, 80, 75]에 대해서 예측값 y를 리턴받아서 pred_y에 저장
pred_y = model(new_var)
print("훈련 후 입력이 73, 80, 75일 때의 예측값 :", pred_y)

훈련 후 입력이 73, 80, 75일 때의 예측값 : tensor([[157.2804]], grad_fn=<AddmmBackward0>)


## 커스텀 데이터셋 (Custom Dataset)

`torch.utils.data.Dataset`을 상속받아 직접 커스텀 데이터셋을 만들 수도 있다. `Dataset`은 데이터셋을 제공하는 추상 클래스이고, 다음 3개의 메소드를 오버라이드한다.

```python
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self):
        # 데이터셋의 전처리를 해주는 부분

    def __len__(self):
        # 데이터셋의 길이. 즉, 총 샘플의 수를 적어주는 부분

    def __getitem__(self, idx):
        # 데이터셋에서 특정 1개의 샘플을 가져오는 함수
```

- `__len__` : `len(dataset)`을 했을 때 데이터셋의 크기를 리턴
- `__getitem__` : `dataset[i]`을 했을 때 i번째 샘플을 가져오도록 하는 인덱싱

## 커스텀 데이터셋으로 선형 회귀 구현하기

`Dataset` 클래스를 상속받아 사용자 정의 데이터셋을 만든다. x_data는 입력 데이터, y_data는 그에 대응하는 출력 데이터다. `__getitem__`은 특정 인덱스에 해당하는 데이터를 `torch.FloatTensor` 형식으로 변환하여 반환한다.

In [69]:
import torch
import torch.nn.functional as F

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [70]:
# Dataset 상속
class CustomDataset(Dataset):
    def __init__(self):
        self.x_data = [[73, 80, 75],
                       [93, 88, 93],
                       [89, 91, 90],
                       [96, 98, 100],
                       [73, 66, 70]]
        self.y_data = [[152], [185], [180], [196], [142]]

    # 총 데이터의 개수를 리턴
    def __len__(self):
        return len(self.x_data)

    # 인덱스를 입력받아 그에 맵핑되는 입출력 데이터를 파이토치의 Tensor 형태로 리턴
    def __getitem__(self, idx):
        x = torch.FloatTensor(self.x_data[idx])
        y = torch.FloatTensor(self.y_data[idx])
        return x, y

In [71]:
dataset = CustomDataset()
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

입력 차원이 3이고 출력 차원이 1인 선형 회귀 모델과, 확률적 경사 하강법(SGD) 옵티마이저를 정의한다.

In [72]:
model = torch.nn.Linear(3,1)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-5)

2,000 에포크 대신 여기서는 20 에포크 동안 미니 배치 학습을 진행한다.

In [73]:
nb_epochs = 20
for epoch in range(nb_epochs + 1):
    for batch_idx, samples in enumerate(dataloader):
        # print(batch_idx)
        # print(samples)
        x_train, y_train = samples
        # H(x) 계산
        prediction = model(x_train)

        # cost 계산
        cost = F.mse_loss(prediction, y_train)

        # cost로 H(x) 계산
        optimizer.zero_grad()
        cost.backward()
        optimizer.step()

        print('Epoch {:4d}/{} Batch {}/{} Cost: {:.6f}'.format(
            epoch, nb_epochs, batch_idx+1, len(dataloader),
            cost.item()
        ))

Epoch    0/20 Batch 1/3 Cost: 40548.742188
Epoch    0/20 Batch 2/3 Cost: 7291.494629
Epoch    0/20 Batch 3/3 Cost: 1811.472290
Epoch    1/20 Batch 1/3 Cost: 1189.893066
Epoch    1/20 Batch 2/3 Cost: 401.466431
Epoch    1/20 Batch 3/3 Cost: 98.862206
Epoch    2/20 Batch 1/3 Cost: 59.924492
Epoch    2/20 Batch 2/3 Cost: 5.559007
Epoch    2/20 Batch 3/3 Cost: 0.005015
Epoch    3/20 Batch 1/3 Cost: 7.238708
Epoch    3/20 Batch 2/3 Cost: 0.579787
Epoch    3/20 Batch 3/3 Cost: 9.432348
Epoch    4/20 Batch 1/3 Cost: 2.596285
Epoch    4/20 Batch 2/3 Cost: 6.112614
Epoch    4/20 Batch 3/3 Cost: 0.173577
Epoch    5/20 Batch 1/3 Cost: 0.585848
Epoch    5/20 Batch 2/3 Cost: 3.764531
Epoch    5/20 Batch 3/3 Cost: 4.496256
Epoch    6/20 Batch 1/3 Cost: 1.750809
Epoch    6/20 Batch 2/3 Cost: 2.692096
Epoch    6/20 Batch 3/3 Cost: 4.092447
Epoch    7/20 Batch 1/3 Cost: 0.873122
Epoch    7/20 Batch 2/3 Cost: 3.545633
Epoch    7/20 Batch 3/3 Cost: 4.466667
Epoch    8/20 Batch 1/3 Cost: 2.943422
Epoch   

In [74]:
# 임의의 입력 [73, 80, 75]를 선언
new_var = torch.FloatTensor([[73, 80, 75]])
# 입력한 값 [73, 80, 75]에 대해서 예측값 y를 리턴받아서 pred_y에 저장
pred_y = model(new_var)
print("훈련 후 입력이 73, 80, 75일 때의 예측값 :", pred_y)

훈련 후 입력이 73, 80, 75일 때의 예측값 : tensor([[153.5625]], grad_fn=<AddmmBackward0>)


# 5. 벡터와 행렬 연산 복습하기

각 변수들의 연산을 **벡터와 행렬 연산**으로 이해할 수 있어야 한다. 즉 데이터와 변수의 개수로부터 행렬·텐서의 크기를 산정할 수 있어야 한다. 여기서 기본적인 벡터와 행렬 연산을 복습한다.

## 벡터와 행렬과 텐서

- **벡터** : 크기와 방향을 가진 양. 숫자가 나열된 형상이며 파이썬에서는 1차원 배열/리스트로 표현한다.
- **행렬** : 행과 열을 가지는 2차원 구조. 가로줄을 행(row), 세로줄을 열(column)이라 한다. 파이썬에서는 2차원 배열로 표현한다.
- **텐서** : 3차원부터는 주로 텐서라고 부른다. 파이썬에서는 3차원 이상의 배열로 표현한다.

## 텐서 (Tensor)

인공 신경망은 복잡한 연산을 주로 행렬 연산으로 해결하는데, 이는 단순히 2차원 배열 연산만을 의미하지 않는다. 입·출력이 복잡해지면 3차원 텐서에 대한 이해가 필수다(예: RNN). Numpy로 텐서를 설명해본다.

> Numpy의 `ndim`을 출력했을 때 나오는 값을 **축(axis)의 개수** 또는 **텐서의 차원**이라 부른다.

### 0차원 텐서 (스칼라)

하나의 실수값으로 이루어진 데이터를 스칼라, 즉 0차원 텐서(0D 텐서)라고 한다.

In [75]:
import numpy as np
d = np.array(5)
print('텐서의 차원 :',d.ndim)
print('텐서의 크기(shape) :',d.shape)

텐서의 차원 : 0
텐서의 크기(shape) : ()


### 1차원 텐서 (벡터)

숫자를 배열한 것을 벡터라 하며, 1차원 텐서(1D 텐서)다. 주의할 점은 **벡터의 차원**과 **텐서의 차원**은 다른 개념이라는 것이다. 아래 예제는 4차원 벡터이지만 1차원 텐서다.

> 벡터에서의 차원은 하나의 축에 놓인 원소의 개수를 의미하고, 텐서에서의 차원은 축의 개수를 의미한다.

In [76]:
d = np.array([1, 2, 3, 4])
print('텐서의 차원 :',d.ndim)
print('텐서의 크기(shape) :',d.shape)

텐서의 차원 : 1
텐서의 크기(shape) : (4,)


### 2차원 텐서 (행렬)

행과 열이 존재하는 벡터의 배열, 즉 행렬(matrix)을 2차원 텐서(2D 텐서)라고 한다.

In [77]:
# 3행 4열의 행렬
d = np.array([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]])
print('텐서의 차원 :',d.ndim)
print('텐서의 크기(shape) :',d.shape)

텐서의 차원 : 2
텐서의 크기(shape) : (3, 4)


### 3차원 텐서 (다차원 배열)

행렬(2차원 텐서)을 단위로 한 번 더 배열하면 3차원 텐서(3D 텐서)다. 3차원 이상부터 본격적으로 텐서라고 부른다.

In [78]:
d = np.array([
            [[1, 2, 3, 4, 5], [6, 7, 8, 9, 10], [10, 11, 12, 13, 14]],
            [[15, 16, 17, 18, 19], [19, 20, 21, 22, 23], [23, 24, 25, 26, 27]]
            ])
print('텐서의 차원 :',d.ndim)
print('텐서의 크기(shape) :',d.shape)

텐서의 차원 : 3
텐서의 크기(shape) : (2, 3, 5)


자연어 처리에서 특히 자주 보게 되는 것이 3D 텐서다. 시퀀스 데이터(주로 단어의 시퀀스 = 문장/문서)를 표현할 때 자주 쓰이기 때문이다. 이 경우 3D 텐서는 **(samples, timesteps, word_dim)** 또는 배치 개념으로 **(batch_size, timesteps, word_dim)**이 된다. samples/batch_size는 샘플의 개수, timesteps는 시퀀스의 길이, word_dim은 단어를 표현하는 벡터의 차원을 의미한다.

예를 들어 다음 3개의 문서를 원-핫 인코딩으로 벡터화한다고 하자.

```
문서1 : I like NLP
문서2 : I like DL
문서3 : DL is AI
```

```
단어   One-hot vector
I    [1 0 0 0 0 0]
like [0 1 0 0 0 0]
NLP  [0 0 1 0 0 0]
DL   [0 0 0 1 0 0]
is   [0 0 0 0 1 0]
AI   [0 0 0 0 0 1]
```

훈련 데이터의 단어들을 모두 원-핫 벡터로 바꿔 한꺼번에 입력으로 사용하면 (3, 3, 6) 크기의 3D 텐서가 된다. 이렇게 훈련 데이터를 다수 묶어 입력으로 사용하는 것을 **배치(Batch)**라고 한다.

### 그 이상의 텐서

3차원 텐서를 배열로 합치면 4차원 텐서, 4차원 텐서를 합치면 5차원 텐서가 된다. 텐서는 다차원 배열로서 계속 확장될 수 있다.

### PyTorch에서의 텐서

2챕터의 '텐서 조작하기(Tensor Manipulation)' 실습을 참고.

## 벡터와 행렬의 연산

벡터와 행렬의 기본적인 연산을 알아본다.

In [79]:
import numpy as np

### 벡터와 행렬의 덧셈과 뺄셈

같은 크기의 두 벡터나 행렬은 덧셈·뺄셈이 가능하다. 같은 위치의 원소끼리 연산하면 되는데, 이를 **요소별(element-wise) 연산**이라 한다.

In [80]:
A = np.array([8, 4, 5])
B = np.array([1, 2, 3])
print('두 벡터의 합 :',A+B)
print('두 벡터의 차 :',A-B)

두 벡터의 합 : [9 6 8]
두 벡터의 차 : [7 2 2]


행렬도 마찬가지다.

In [81]:
A = np.array([[10, 20, 30, 40], [50, 60, 70, 80]])
B = np.array([[5, 6, 7, 8],[1, 2, 3, 4]])
print('두 행렬의 합 :')
print(A + B)
print('두 행렬의 차 :')
print(A - B)

두 행렬의 합 :
[[15 26 37 48]
 [51 62 73 84]]
두 행렬의 차 :
[[ 5 14 23 32]
 [49 58 67 76]]


### 벡터의 내적과 행렬의 곱셈

벡터의 점곱(dot product) 또는 내적(inner product)은 연산을 점(dot)으로 표현해 $a \cdot b$ 와 같이 나타낸다. 내적이 성립하려면 두 벡터의 차원이 같아야 하고, 앞의 벡터가 행벡터(가로), 뒤의 벡터가 열벡터(세로)여야 한다. **벡터의 내적의 결과는 스칼라**가 된다. 예) $[1, 2, 3] \cdot [4, 5, 6] = 1{\times}4 + 2{\times}5 + 3{\times}6 = 32$

In [82]:
A = np.array([1, 2, 3])
B = np.array([4, 5, 6])
print('두 벡터의 내적 :',np.dot(A, B))

두 벡터의 내적 : 32


행렬의 곱셈은 왼쪽 행렬의 행벡터와 오른쪽 행렬의 열벡터의 내적이 결과 행렬의 원소가 되는 것으로 이루어진다.

In [83]:
A = np.array([[1, 3],[2, 4]])
B = np.array([[5, 7],[6, 8]])
print('두 행렬의 행렬곱 :')
print(np.matmul(A, B))

두 행렬의 행렬곱 :
[[23 31]
 [34 46]]


> 행렬 곱셈의 두 가지 주요한 조건 (반드시 기억)
> - 두 행렬의 곱 A × B가 성립하려면, A의 **열**의 개수와 B의 **행**의 개수가 같아야 한다.
> - 곱의 결과로 나온 행렬 AB의 크기는 A의 **행**의 개수와 B의 **열**의 개수를 가진다.

## 다중 선형 회귀 행렬 연산으로 이해하기

독립 변수가 n개인 다중 선형 회귀 수식 $y = w_1 x_1 + \cdots + w_n x_n + b$ 는 입력 벡터와 가중치 벡터의 내적으로 표현할 수 있다. 샘플이 많을 경우에는 행렬의 곱셈으로 표현한다. 입력 행렬 X와 가중치 W, 편향 B에 대해 전체 가설은 다음과 같다.

$$ H(X) = XW + B $$

예를 들어 입력 행렬 X가 5행 4열이고 출력 벡터 Y가 5행 1열이라면, 행렬곱이 성립하기 위해 가중치 W의 크기는 4행 1열이어야 함을 추론할 수 있다. (수학적 관례로는 가중치 W가 입력 X 앞에 오는 $H(X) = WX + B$ 로 표현하기도 한다.)

인공 신경망도 본질적으로 위와 같은 행렬 연산이다.

## 샘플 (Sample)과 특성 (Feature)

훈련 데이터의 입력 행렬을 X라 할 때, 머신 러닝에서 데이터를 셀 수 있는 단위로 구분한 각각을 **샘플(Sample)**이라 하고, 종속 변수 y를 예측하기 위한 각각의 독립 변수 x를 **특성(Feature)**이라 한다.

## 가중치와 편향 행렬의 크기 결정

앞서 본 행렬 곱셈의 두 조건을 이용하면, 입력 행렬과 출력 행렬의 크기로부터 가중치 행렬 W와 편향 행렬 B의 크기를 찾아낼 수 있다. 입력 행렬을 X, 출력 행렬을 Y라고 할 때 다음과 같이 추론한다.

$$ X_{m \times n} \times W_{? \times ?} + B_{? \times ?} = Y_{m \times j} $$

- 행렬의 덧셈에 해당하는 **B**는 Y의 크기에 영향을 주지 않으므로, **B의 크기는 Y의 크기와 같다** → $B_{m \times j}$
- 행렬곱이 성립하려면 앞 행렬의 열 크기와 뒤 행렬의 행 크기가 같아야 하므로, 입력 X로부터 **W의 행 크기가 결정**된다 → $W_{n \times ?}$
- 곱의 결과로 나온 행렬의 열 크기는 뒤 행렬의 열 크기와 같으므로, 출력 Y로부터 **W의 열 크기가 결정**된다 → $W_{n \times j}$

입력·출력 행렬의 크기로부터 가중치·편향 행렬의 크기를 추정할 수 있으면, 모델에 존재하는 **총 매개변수의 개수**(가중치·편향 행렬의 모든 원소의 수)를 계산하기 쉽다.